#### Importing Libraries

In [1]:
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.statespace.sarimax import SARIMAX

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

sns.set_theme(style="whitegrid")

print("Model Evaluation Libraries Imported Successfully")

Model Evaluation Libraries Imported Successfully


#### Configuring Project Paths

In [2]:
project_dir = Path("..").resolve()

data_dir = project_dir / "data" / "processed"
models_dir = project_dir / "models"
artifacts_dir = project_dir / "artifacts"
reports_dir = project_dir / "reports"
figures_dir = reports_dir / "figures"

figures_dir.mkdir(parents=True, exist_ok=True)

required_files = {
    "Daily Revenue": data_dir / "daily_revenue.parquet",
    "Training Features": data_dir / "forecast_features_train.parquet",
    "Test Features": data_dir / "forecast_features_test.parquet",
    "Forecast Feature Configuration": artifacts_dir / "forecast_feature_config.json",
    "Final Forecasting Configuration": artifacts_dir / "final_forecasting_configuration.json",
    "AutoReg Model": models_dir / "final_autoreg.pkl",
    "SARIMA Model": models_dir / "final_sarima.pkl",
    "Extra Trees Models": models_dir / "final_extra_trees.joblib"
}

file_verification = pd.DataFrame([
    {
        "Artifact": name,
        "Exists": path.is_file(),
        "Path": str(path.relative_to(project_dir))
    }
    for name, path in required_files.items()
])

display(file_verification)

assert file_verification["Exists"].all(), (
    "One or more required evaluation artifacts are missing."
)

print("\nProject Directory:", project_dir)
print("Required Artifacts:", len(required_files))
print("Artifact Verification: Completed")

,Artifact,Exists,Path
0,Daily Revenue,True,data\processed\daily_revenue.parquet
1,Training Features,True,data\processed\forecast_features_train.parquet
2,Test Features,True,data\processed\forecast_features_test.parquet
3,Forecast Feature Configuration,True,artifacts\forecast_feature_config.json
4,Final Forecasting Configuration,True,artifacts\final_forecasting_configuration.json
5,AutoReg Model,True,models\final_autoreg.pkl
6,SARIMA Model,True,models\final_sarima.pkl
7,Extra Trees Models,True,models\final_extra_trees.joblib



Project Directory: D:\Projects\ML and Data Science\Retail Sales Dynamics
Required Artifacts: 8
Artifact Verification: Completed


**Key Observations**
- **Artifact Availability:** All eight required datasets, forecasting configurations, and fitted model artifacts were successfully located within the project directory.
- **Evaluation Infrastructure:** The project paths and reporting directories are configured, providing access to the saved forecasting components without repeating model development.

#### Loading Evaluation Datasets and Configurations

In [3]:
daily_revenue_df = pd.read_parquet(
    data_dir / "daily_revenue.parquet"
)

train_df = pd.read_parquet(
    data_dir / "forecast_features_train.parquet"
)

test_df = pd.read_parquet(
    data_dir / "forecast_features_test.parquet"
)

with open(
    artifacts_dir / "forecast_feature_config.json",
    "r",
    encoding="utf-8"
) as file:
    feature_config = json.load(file)

with open(
    artifacts_dir / "final_forecasting_configuration.json",
    "r",
    encoding="utf-8"
) as file:
    final_model_config = json.load(file)

dataset_summary_df = pd.DataFrame([
    {
        "Dataset": "Daily Revenue",
        "Rows": len(daily_revenue_df),
        "Columns": daily_revenue_df.shape[1]
    },
    {
        "Dataset": "Training Features",
        "Rows": len(train_df),
        "Columns": train_df.shape[1]
    },
    {
        "Dataset": "Test Features",
        "Rows": len(test_df),
        "Columns": test_df.shape[1]
    }
])

print("Evaluation Dataset Summary:")
display(dataset_summary_df)

print("\nDaily Revenue Columns:")
print(daily_revenue_df.columns.tolist())

print("\nTraining Feature Columns (First 10):")
print(train_df.columns.tolist()[:10])

print("\nTest Feature Columns (First 10):")
print(test_df.columns.tolist()[:10])

print("\nFeature Configuration Keys:")
print(list(feature_config.keys()))

print("\nFinal Model Configuration Keys:")
print(list(final_model_config.keys()))

print("\nFinal Model:", final_model_config["model_name"])
print("Forecast Horizon:", final_model_config["forecast_horizon"])

print("\nEvaluation Datasets and Configurations: Loaded Successfully")

Evaluation Dataset Summary:


,Dataset,Rows,Columns
0,Daily Revenue,305,2
1,Training Features,207,87
2,Test Features,55,87



Daily Revenue Columns:
['Transaction_Date', 'Daily_Revenue']

Training Feature Columns (First 10):
['Forecast_Origin', 'Origin_Revenue', 'Target_H1', 'Target_Date_H1', 'Target_H2', 'Target_Date_H2', 'Target_H3', 'Target_Date_H3', 'Target_H4', 'Target_Date_H4']

Test Feature Columns (First 10):
['Forecast_Origin', 'Origin_Revenue', 'Target_H1', 'Target_Date_H1', 'Target_H2', 'Target_Date_H2', 'Target_H3', 'Target_Date_H3', 'Target_H4', 'Target_Date_H4']

Feature Configuration Keys:
['forecast_horizon', 'forecasting_approach', 'horizon_unit', 'training_end_date', 'future_trading_dates_assumed_known', 'shared_features', 'horizon_features', 'horizon_targets']

Final Model Configuration Keys:
['model_name', 'forecast_horizon', 'forecast_unit', 'target', 'ensemble_method', 'components', 'validation_mae', 'validation_rmse', 'validation_r2', 'training_end_date', 'random_seed']

Final Model: Voting_Hybrid
Forecast Horizon: 7

Evaluation Datasets and Configurations: Loaded Successfully


**Key Observations**
- **Evaluation Dataset Structure:** The daily revenue dataset contains 305 observed trading days, while the engineered training and test datasets contain 207 and 55 forecast origins, respectively, with 87 columns each.
- **Forecasting Configuration:** The saved configurations confirm a seven-trading-day forecasting horizon and the selected Voting Hybrid ensemble, providing the necessary structure for consistent test-set evaluation.

#### Validating the Evaluation Timeline and Forecast Horizons

In [5]:
daily_revenue_df["Transaction_Date"] = pd.to_datetime(
    daily_revenue_df["Transaction_Date"]
)

train_df["Forecast_Origin"] = pd.to_datetime(
    train_df["Forecast_Origin"]
)

test_df["Forecast_Origin"] = pd.to_datetime(
    test_df["Forecast_Origin"]
)

forecast_horizon = int(feature_config["forecast_horizon"])
training_end_date = pd.Timestamp(
    feature_config["training_end_date"]
)

for horizon in range(1, forecast_horizon + 1):
    target_date_column = f"Target_Date_H{horizon}"

    train_df[target_date_column] = pd.to_datetime(
        train_df[target_date_column]
    )

    test_df[target_date_column] = pd.to_datetime(
        test_df[target_date_column]
    )

timeline_summary_df = pd.DataFrame([
    {
        "Dataset": "Training",
        "Forecast_Origins": len(train_df),
        "First_Origin": train_df["Forecast_Origin"].min(),
        "Last_Origin": train_df["Forecast_Origin"].max(),
        "First_Target_H1": train_df["Target_Date_H1"].min(),
        "Last_Target_H7": train_df["Target_Date_H7"].max()
    },
    {
        "Dataset": "Test",
        "Forecast_Origins": len(test_df),
        "First_Origin": test_df["Forecast_Origin"].min(),
        "Last_Origin": test_df["Forecast_Origin"].max(),
        "First_Target_H1": test_df["Target_Date_H1"].min(),
        "Last_Target_H7": test_df["Target_Date_H7"].max()
    }
])

print("Training Cutoff:", training_end_date.date())

print("\nForecasting Timeline Summary:")
display(timeline_summary_df)

print("\nFirst Five Test Forecast Origins:")
display(
    test_df[
        ["Forecast_Origin", "Target_Date_H1", "Target_Date_H7"]
    ].head()
)

print("\nTest Origins on or Before Training Cutoff:")
print(
    (
        test_df["Forecast_Origin"] <= training_end_date
    ).sum()
)

print("\nTest Targets on or Before Training Cutoff:")

for horizon in range(1, forecast_horizon + 1):
    target_date_column = f"Target_Date_H{horizon}"

    count = (
        test_df[target_date_column] <= training_end_date
    ).sum()

    print(f"Horizon {horizon}: {count}")

Training Cutoff: 2011-09-29

Forecasting Timeline Summary:


,Dataset,Forecast_Origins,First_Origin,Last_Origin,First_Target_H1,Last_Target_H7
0,Training,207,2011-01-16,2011-09-21,2011-01-17,2011-09-29
1,Test,55,2011-09-29,2011-12-01,2011-09-30,2011-12-09



First Five Test Forecast Origins:


,Forecast_Origin,Target_Date_H1,Target_Date_H7
0,2011-09-29,2011-09-30,2011-10-07
1,2011-09-30,2011-10-02,2011-10-09
2,2011-10-02,2011-10-03,2011-10-10
3,2011-10-03,2011-10-04,2011-10-11
4,2011-10-04,2011-10-05,2011-10-12



Test Origins on or Before Training Cutoff:
1

Test Targets on or Before Training Cutoff:
Horizon 1: 0
Horizon 2: 0
Horizon 3: 0
Horizon 4: 0
Horizon 5: 0
Horizon 6: 0
Horizon 7: 0


**Key Observations**
- **Chronological Separation:** Training forecast targets end on September 29, 2011, while test targets begin on September 30, 2011, confirming that the evaluation targets do not overlap the training period.
- **Forecast Origin Alignment:** The first test forecast originates on the training cutoff, with all seven forecast horizons targeting subsequent trading days, supporting the intended rolling-origin evaluation framework.

#### Loading the Final Forecasting Models

In [6]:
final_autoreg_model = joblib.load(
    models_dir / "final_autoreg.pkl"
)

final_sarima_model = joblib.load(
    models_dir / "final_sarima.pkl"
)

final_extra_trees_models = joblib.load(
    models_dir / "final_extra_trees.joblib"
)

horizon_feature_columns = {
    int(horizon): columns
    for horizon, columns in feature_config["horizon_features"].items()
}

horizon_target_columns = {
    horizon: f"Target_H{horizon}"
    for horizon in range(1, forecast_horizon + 1)
}

assert final_model_config["model_name"] == "Voting_Hybrid"
assert final_model_config["forecast_horizon"] == forecast_horizon

assert set(final_model_config["components"]) == {
    "AutoReg",
    "SARIMA",
    "Extra_Trees"
}

assert len(final_extra_trees_models) == forecast_horizon

model_verification_records = []

for horizon in range(1, forecast_horizon + 1):
    model = final_extra_trees_models[horizon]
    feature_columns = horizon_feature_columns[horizon]

    assert all(
        column in test_df.columns
        for column in feature_columns
    )

    assert model.n_features_in_ == len(feature_columns)

    model_verification_records.append({
        "Horizon": horizon,
        "Features": len(feature_columns),
        "Model_Type": type(model).__name__,
        "Status": "Verified"
    })

model_verification_df = pd.DataFrame(
    model_verification_records
)

print("Final Forecasting Models:")
print("AutoReg:", type(final_autoreg_model).__name__)
print("SARIMA:", type(final_sarima_model).__name__)
print("Ensemble:", final_model_config["model_name"])

print("\nExtra Trees Horizon Verification:")
display(model_verification_df)

print("\nFinal Forecasting Models: Loaded Successfully")

Final Forecasting Models:
AutoReg: AutoRegResultsWrapper
SARIMA: SARIMAXResultsWrapper
Ensemble: Voting_Hybrid

Extra Trees Horizon Verification:


,Horizon,Features,Model_Type,Status
0,1,48,ExtraTreesRegressor,Verified
1,2,48,ExtraTreesRegressor,Verified
2,3,48,ExtraTreesRegressor,Verified
3,4,48,ExtraTreesRegressor,Verified
4,5,48,ExtraTreesRegressor,Verified
5,6,48,ExtraTreesRegressor,Verified
6,7,48,ExtraTreesRegressor,Verified



Final Forecasting Models: Loaded Successfully


**Key Observations**
- **Final Model Availability:** The fitted AutoReg, SARIMA, and Extra Trees models were successfully loaded, confirming that all three Voting Hybrid components are available for test-set forecasting.
- **Horizon-Specific Compatibility:** All seven Extra Trees models passed feature compatibility checks, with each model expecting the 48 engineered predictors defined for its respective forecast horizon.

#### Preparing the Rolling-Origin Evaluation Framework

In [8]:
daily_revenue_series = (
    daily_revenue_df
    .sort_values("Transaction_Date")
    .set_index("Transaction_Date")["Daily_Revenue"]
    .astype(float)
)

trading_dates = pd.DatetimeIndex(
    daily_revenue_series.index
)

trading_day_positions = {
    date: position
    for position, date in enumerate(trading_dates)
}

test_origin_positions = test_df["Forecast_Origin"].map(
    trading_day_positions
)

assert test_origin_positions.notna().all()

test_origin_positions = test_origin_positions.astype(int)

assert test_origin_positions.is_monotonic_increasing

evaluation_records = []

for row_index, row in test_df.iterrows():
    origin_date = row["Forecast_Origin"]
    origin_position = trading_day_positions[origin_date]

    available_history = daily_revenue_series.iloc[
        :origin_position + 1
    ]

    assert available_history.index.max() == origin_date

    for horizon in range(1, forecast_horizon + 1):
        target_date = row[f"Target_Date_H{horizon}"]
        expected_date = trading_dates[
            origin_position + horizon
        ]

        assert target_date == expected_date
        assert target_date > origin_date
        assert target_date > training_end_date

        evaluation_records.append({
            "Forecast_Origin": origin_date,
            "Origin_Position": origin_position,
            "Horizon": horizon,
            "Target_Date": target_date,
            "Actual": row[f"Target_H{horizon}"],
            "History_Length": len(available_history)
        })

rolling_evaluation_df = pd.DataFrame(
    evaluation_records
)

assert len(rolling_evaluation_df) == (
    len(test_df) * forecast_horizon
)

assert rolling_evaluation_df["Actual"].notna().all()

print("Rolling-Origin Evaluation Summary:")
print("Test Forecast Origins:", len(test_df))
print("Forecast Horizon:", forecast_horizon)
print("Total Forecast Evaluations:", len(rolling_evaluation_df))
print("First Forecast Origin:", test_df["Forecast_Origin"].min().date())
print("Last Forecast Origin:", test_df["Forecast_Origin"].max().date())

print("\nAvailable History:")
print(
    "Minimum Observations:",
    rolling_evaluation_df["History_Length"].min()
)
print(
    "Maximum Observations:",
    rolling_evaluation_df["History_Length"].max()
)

print("\nFirst Five Evaluation Records:")
display(rolling_evaluation_df.head())

print("\nRolling-Origin Evaluation Framework: Prepared Successfully")

Rolling-Origin Evaluation Summary:
Test Forecast Origins: 55
Forecast Horizon: 7
Total Forecast Evaluations: 385
First Forecast Origin: 2011-09-29
Last Forecast Origin: 2011-12-01

Available History:
Minimum Observations: 244
Maximum Observations: 298

First Five Evaluation Records:


,Forecast_Origin,Origin_Position,Horizon,Target_Date,Actual,History_Length
0,2011-09-29,243,1,2011-09-30,42791.13,244
1,2011-09-29,243,2,2011-10-02,11623.58,244
2,2011-09-29,243,3,2011-10-03,60423.71,244
3,2011-09-29,243,4,2011-10-04,47135.98,244
4,2011-09-29,243,5,2011-10-05,74470.06,244



Rolling-Origin Evaluation Framework: Prepared Successfully


**Key Observations**
- **Rolling-Origin Evaluation:** The test framework contains 55 forecast origins and seven forecasting horizons, producing 385 aligned forecast evaluations across the reserved test period.
- **Historical Data Availability:** Available forecasting history increases from 244 to 298 observed trading days, supporting sequential evaluation using revenue observations available at each forecast origin.

#### Generating Extra Trees Test Forecasts

In [9]:
extra_trees_prediction_records = []

for horizon in range(1, forecast_horizon + 1):
    model = final_extra_trees_models[horizon]
    feature_columns = horizon_feature_columns[horizon]

    X_test_horizon = test_df[feature_columns]

    assert X_test_horizon.notna().all().all()
    assert np.isfinite(
        X_test_horizon.to_numpy(dtype=float)
    ).all()

    predictions = model.predict(X_test_horizon)

    assert len(predictions) == len(test_df)
    assert np.isfinite(predictions).all()

    for origin_date, prediction in zip(
        test_df["Forecast_Origin"],
        predictions
    ):
        extra_trees_prediction_records.append({
            "Forecast_Origin": origin_date,
            "Horizon": horizon,
            "Extra_Trees": float(prediction)
        })

extra_trees_predictions_df = pd.DataFrame(
    extra_trees_prediction_records
)

assert len(extra_trees_predictions_df) == len(
    rolling_evaluation_df
)

assert not extra_trees_predictions_df.duplicated(
    subset=["Forecast_Origin", "Horizon"]
).any()

rolling_evaluation_df = rolling_evaluation_df.merge(
    extra_trees_predictions_df,
    on=["Forecast_Origin", "Horizon"],
    how="left",
    validate="one_to_one"
)

assert rolling_evaluation_df["Extra_Trees"].notna().all()

print("Extra Trees Test Forecast Summary:")
print("Forecast Origins:", len(test_df))
print("Forecast Horizons:", forecast_horizon)
print("Total Predictions:", len(extra_trees_predictions_df))

print("\nFirst Five Predictions:")
display(
    rolling_evaluation_df[
        [
            "Forecast_Origin",
            "Horizon",
            "Target_Date",
            "Actual",
            "Extra_Trees"
        ]
    ].head()
)

print("\nExtra Trees Test Forecasts: Generated Successfully")

Extra Trees Test Forecast Summary:
Forecast Origins: 55
Forecast Horizons: 7
Total Predictions: 385

First Five Predictions:


,Forecast_Origin,Horizon,Target_Date,Actual,Extra_Trees
0,2011-09-29,1,2011-09-30,42791.13,44209.001726
1,2011-09-29,2,2011-10-02,11623.58,16183.392451
2,2011-09-29,3,2011-10-03,60423.71,38475.336799
3,2011-09-29,4,2011-10-04,47135.98,38220.300611
4,2011-09-29,5,2011-10-05,74470.06,43352.161197



Extra Trees Test Forecasts: Generated Successfully


**Key Observations**
- **Test Forecast Generation:** All seven horizon-specific Extra Trees models successfully generated 385 predictions across 55 test forecast origins, with complete alignment between forecast origins and horizons.
- **Initial Forecast Behavior:** The sample predictions show varying deviations from actual daily revenue, highlighting the need for comprehensive error analysis across the full test period rather than relying on individual forecasts.

#### Generating Rolling-Origin Statistical Forecasts

In [10]:
autoreg_lags = [1, 2, 3, 6, 12]
sarima_order = (1, 0, 1)
sarima_seasonal_order = (1, 0, 1, 6)

statistical_prediction_records = []

for origin_index, origin_row in enumerate(
    test_df.itertuples(index=False),
    start=1
):
    origin_date = origin_row.Forecast_Origin
    origin_position = trading_day_positions[origin_date]

    available_history = (
        daily_revenue_series
        .iloc[:origin_position + 1]
        .reset_index(drop=True)
        .astype(float)
    )

    assert len(available_history) == origin_position + 1

    autoreg_model = AutoReg(
        available_history,
        lags=autoreg_lags,
        trend="c"
    ).fit()

    autoreg_forecasts = np.asarray(
        autoreg_model.forecast(steps=forecast_horizon),
        dtype=float
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        sarima_model = SARIMAX(
            available_history,
            order=sarima_order,
            seasonal_order=sarima_seasonal_order,
            trend="c",
            enforce_stationarity=False,
            enforce_invertibility=False
        ).fit(
            disp=False,
            maxiter=100
        )

    sarima_forecasts = np.asarray(
        sarima_model.forecast(steps=forecast_horizon),
        dtype=float
    )

    assert len(autoreg_forecasts) == forecast_horizon
    assert len(sarima_forecasts) == forecast_horizon
    assert np.isfinite(autoreg_forecasts).all()
    assert np.isfinite(sarima_forecasts).all()

    for horizon in range(1, forecast_horizon + 1):
        statistical_prediction_records.append({
            "Forecast_Origin": origin_date,
            "Horizon": horizon,
            "AutoReg": float(autoreg_forecasts[horizon - 1]),
            "SARIMA": float(sarima_forecasts[horizon - 1])
        })

    if origin_index % 10 == 0 or origin_index == len(test_df):
        print(
            f"Completed {origin_index}/{len(test_df)} "
            "forecast origins"
        )

statistical_predictions_df = pd.DataFrame(
    statistical_prediction_records
)

assert len(statistical_predictions_df) == (
    len(test_df) * forecast_horizon
)

assert not statistical_predictions_df.duplicated(
    subset=["Forecast_Origin", "Horizon"]
).any()

rolling_evaluation_df = rolling_evaluation_df.merge(
    statistical_predictions_df,
    on=["Forecast_Origin", "Horizon"],
    how="left",
    validate="one_to_one"
)

assert rolling_evaluation_df[
    ["AutoReg", "SARIMA"]
].notna().all().all()

print("\nStatistical Forecast Summary:")
print("Forecast Origins:", len(test_df))
print("Total Predictions per Model:", len(statistical_predictions_df))

print("\nFirst Five Forecasts:")
display(
    rolling_evaluation_df[
        [
            "Forecast_Origin",
            "Horizon",
            "Actual",
            "AutoReg",
            "SARIMA"
        ]
    ].head()
)

print("\nRolling-Origin Statistical Forecasts: Generated Successfully")

Completed 10/55 forecast origins
Completed 20/55 forecast origins
Completed 30/55 forecast origins
Completed 40/55 forecast origins
Completed 50/55 forecast origins
Completed 55/55 forecast origins

Statistical Forecast Summary:
Forecast Origins: 55
Total Predictions per Model: 385

First Five Forecasts:


,Forecast_Origin,Horizon,Actual,AutoReg,SARIMA
0,2011-09-29,1,42791.13,31484.871791,30305.362992
1,2011-09-29,2,11623.58,26614.540589,28385.326171
2,2011-09-29,3,60423.71,31219.542407,28699.637839
3,2011-09-29,4,47135.98,44472.809427,36412.396959
4,2011-09-29,5,74470.06,35466.266915,33296.016053



Rolling-Origin Statistical Forecasts: Generated Successfully


**Key Observations**
- **Statistical Forecast Generation:** AutoReg and SARIMA successfully generated 385 predictions each across 55 rolling forecast origins, using the selected model configurations and progressively available historical observations.
- **Forecasting Behavior:** Initial predictions reveal differences between the two statistical approaches and notable deviations from actual revenue, reinforcing the importance of evaluating performance across all forecast origins and horizons.

#### Generating Voting Hybrid Ensemble Forecasts

In [11]:
ensemble_components = [
    "AutoReg",
    "SARIMA",
    "Extra_Trees"
]

ensemble_weights = np.full(
    len(ensemble_components),
    1.0 / len(ensemble_components)
)

assert final_model_config["model_name"] == "Voting_Hybrid"
assert set(final_model_config["components"]) == set(
    ensemble_components
)

assert np.isclose(ensemble_weights.sum(), 1.0)

assert rolling_evaluation_df[
    ensemble_components
].notna().all().all()

rolling_evaluation_df["Voting_Hybrid"] = np.average(
    rolling_evaluation_df[ensemble_components].to_numpy(
        dtype=float
    ),
    axis=1,
    weights=ensemble_weights
)

assert np.isfinite(
    rolling_evaluation_df["Voting_Hybrid"]
).all()

expected_predictions = (
    rolling_evaluation_df["AutoReg"]
    + rolling_evaluation_df["SARIMA"]
    + rolling_evaluation_df["Extra_Trees"]
) / 3

assert np.allclose(
    rolling_evaluation_df["Voting_Hybrid"],
    expected_predictions
)

ensemble_summary_df = pd.DataFrame({
    "Component": ensemble_components,
    "Weight": ensemble_weights,
    "Weight_Percentage": ensemble_weights * 100
})

print("Voting Hybrid Ensemble Configuration:")
display(ensemble_summary_df)

print("\nTest Forecast Summary:")
print("Forecast Origins:", len(test_df))
print("Forecast Horizons:", forecast_horizon)
print("Total Ensemble Predictions:", len(rolling_evaluation_df))

print("\nFirst Five Ensemble Forecasts:")
display(
    rolling_evaluation_df[
        [
            "Forecast_Origin",
            "Horizon",
            "Target_Date",
            "Actual",
            "AutoReg",
            "SARIMA",
            "Extra_Trees",
            "Voting_Hybrid"
        ]
    ].head()
)

print("\nVoting Hybrid Ensemble Forecasts: Generated Successfully")

Voting Hybrid Ensemble Configuration:


,Component,Weight,Weight_Percentage
0,AutoReg,0.333333,33.333333
1,SARIMA,0.333333,33.333333
2,Extra_Trees,0.333333,33.333333



Test Forecast Summary:
Forecast Origins: 55
Forecast Horizons: 7
Total Ensemble Predictions: 385

First Five Ensemble Forecasts:


,Forecast_Origin,Horizon,Target_Date,Actual,AutoReg,SARIMA,Extra_Trees,Voting_Hybrid
0,2011-09-29,1,2011-09-30,42791.13,31484.871791,30305.362992,44209.001726,35333.078836
1,2011-09-29,2,2011-10-02,11623.58,26614.540589,28385.326171,16183.392451,23727.753070
2,2011-09-29,3,2011-10-03,60423.71,31219.542407,28699.637839,38475.336799,32798.172348
3,2011-09-29,4,2011-10-04,47135.98,44472.809427,36412.396959,38220.300611,39701.835666
4,2011-09-29,5,2011-10-05,74470.06,35466.266915,33296.016053,43352.161197,37371.481388



Voting Hybrid Ensemble Forecasts: Generated Successfully


**Key Observations**
- **Ensemble Forecast Generation:** Voting Hybrid successfully combined predictions from AutoReg, SARIMA, and Extra Trees using equal weights of 33.33%, producing 385 forecasts across 55 test origins and seven horizons.
- **Initial Ensemble Behavior:** The sample forecasts demonstrate how averaging combines the different model predictions, although several high-revenue observations remain substantially underestimated.

#### Evaluating Individual and Ensemble Forecasting Performance

In [12]:
evaluation_models = [
    "AutoReg",
    "SARIMA",
    "Extra_Trees",
    "Voting_Hybrid"
]

actual_values = rolling_evaluation_df["Actual"].to_numpy(
    dtype=float
)

assert len(actual_values) == 385
assert np.isfinite(actual_values).all()

model_evaluation_records = []

for model_name in evaluation_models:
    predicted_values = rolling_evaluation_df[
        model_name
    ].to_numpy(dtype=float)

    assert len(predicted_values) == len(actual_values)
    assert np.isfinite(predicted_values).all()

    model_evaluation_records.append({
        "Model": model_name,
        "MAE": mean_absolute_error(
            actual_values,
            predicted_values
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                actual_values,
                predicted_values
            )
        ),
        "R2": r2_score(
            actual_values,
            predicted_values
        ),
        "Forecasts": len(predicted_values)
    })

test_model_performance_df = (
    pd.DataFrame(model_evaluation_records)
    .sort_values("MAE")
    .reset_index(drop=True)
)

test_model_performance_df.insert(
    0,
    "Rank",
    range(1, len(test_model_performance_df) + 1)
)

print("Test-Set Forecasting Performance:")
display(
    test_model_performance_df.style.format({
        "MAE": "{:,.2f}",
        "RMSE": "{:,.2f}",
        "R2": "{:.4f}"
    })
)

best_test_model = test_model_performance_df.iloc[0]["Model"]
voting_hybrid_test_mae = test_model_performance_df.loc[
    test_model_performance_df["Model"] == "Voting_Hybrid",
    "MAE"
].iloc[0]

print("\nBest Test Model:", best_test_model)
print("Voting Hybrid Test MAE:", round(voting_hybrid_test_mae, 4))
print("Total Forecast Evaluations:", len(actual_values))

print("\nTest-Set Forecasting Performance: Evaluated Successfully")

Test-Set Forecasting Performance:


,Rank,Model,MAE,RMSE,R2,Forecasts
0,1,Extra_Trees,"12,374.70","16,749.57",0.1244,385
1,2,Voting_Hybrid,"12,465.39","17,080.80",0.0894,385
2,3,AutoReg,"13,303.88","17,386.78",0.0565,385
3,4,SARIMA,"14,131.27","18,703.35",-0.0918,385



Best Test Model: Extra_Trees
Voting Hybrid Test MAE: 12465.3857
Total Forecast Evaluations: 385

Test-Set Forecasting Performance: Evaluated Successfully


**Key Observations**
- **Test-Set Model Performance:** Extra Trees achieved the lowest test MAE of 12,374.70 and RMSE of 16,749.57, outperforming Voting Hybrid, AutoReg, and SARIMA across 385 forecast evaluations.
- **Ensemble Generalization:** Voting Hybrid recorded a test MAE of 12,465.39, compared with its validation MAE of 11,690.95, indicating a modest decline in forecasting accuracy on unseen observations while remaining the second-best evaluated model.

#### Comparing Forecasting Models Against Baselines

In [13]:
seasonal_period = 6

baseline_evaluation_df = rolling_evaluation_df.copy()

naive_predictions = []
seasonal_naive_predictions = []

for row in baseline_evaluation_df.itertuples(index=False):
    origin_position = int(row.Origin_Position)
    horizon = int(row.Horizon)

    naive_prediction = daily_revenue_series.iloc[
        origin_position
    ]

    seasonal_reference_position = (
        origin_position
        + horizon
        - seasonal_period
    )

    while seasonal_reference_position > origin_position:
        seasonal_reference_position -= seasonal_period

    assert seasonal_reference_position >= 0
    assert seasonal_reference_position <= origin_position

    seasonal_prediction = daily_revenue_series.iloc[
        seasonal_reference_position
    ]

    naive_predictions.append(float(naive_prediction))
    seasonal_naive_predictions.append(
        float(seasonal_prediction)
    )

baseline_evaluation_df["Naive"] = naive_predictions
baseline_evaluation_df["Seasonal_Naive"] = (
    seasonal_naive_predictions
)

baseline_models = [
    "Naive",
    "Seasonal_Naive"
]

baseline_performance_records = []

for model_name in baseline_models:
    predictions = baseline_evaluation_df[
        model_name
    ].to_numpy(dtype=float)

    baseline_performance_records.append({
        "Model": model_name,
        "MAE": mean_absolute_error(
            actual_values,
            predictions
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                actual_values,
                predictions
            )
        ),
        "R2": r2_score(
            actual_values,
            predictions
        ),
        "Forecasts": len(predictions)
    })

baseline_performance_df = pd.DataFrame(
    baseline_performance_records
)

seasonal_naive_test_mae = baseline_performance_df.loc[
    baseline_performance_df["Model"] == "Seasonal_Naive",
    "MAE"
].iloc[0]

combined_test_performance_df = pd.concat(
    [
        test_model_performance_df.drop(columns="Rank"),
        baseline_performance_df
    ],
    ignore_index=True
)

combined_test_performance_df[
    "Improvement_vs_Seasonal_Naive_%"
] = (
    (
        seasonal_naive_test_mae
        - combined_test_performance_df["MAE"]
    )
    / seasonal_naive_test_mae
    * 100
)

combined_test_performance_df = (
    combined_test_performance_df
    .sort_values("MAE")
    .reset_index(drop=True)
)

combined_test_performance_df.insert(
    0,
    "Rank",
    range(1, len(combined_test_performance_df) + 1)
)

print("Test-Set Model and Baseline Comparison:")

display(
    combined_test_performance_df.style.format({
        "MAE": "{:,.2f}",
        "RMSE": "{:,.2f}",
        "R2": "{:.4f}",
        "Improvement_vs_Seasonal_Naive_%": "{:+.2f}%"
    })
)

print("\nSeasonal Naive Test MAE:", round(
    seasonal_naive_test_mae, 4
))

print("\nModel and Baseline Comparison: Completed")

Test-Set Model and Baseline Comparison:


,Rank,Model,MAE,RMSE,R2,Forecasts,Improvement_vs_Seasonal_Naive_%
0,1,Extra_Trees,"12,374.70","16,749.57",0.1244,385,+11.23%
1,2,Voting_Hybrid,"12,465.39","17,080.80",0.0894,385,+10.58%
2,3,AutoReg,"13,303.88","17,386.78",0.0565,385,+4.57%
3,4,Seasonal_Naive,"13,940.46","18,311.69",-0.0466,385,+0.00%
4,5,SARIMA,"14,131.27","18,703.35",-0.0918,385,-1.37%
5,6,Naive,"18,349.33","23,467.37",-0.7188,385,-31.63%



Seasonal Naive Test MAE: 13940.4638

Model and Baseline Comparison: Completed


**Key Observations**
- **Baseline Improvement:** Extra Trees achieved the strongest test performance, reducing MAE by 11.23% relative to Seasonal Naive, followed closely by Voting Hybrid with a 10.58% improvement.
- **Model Effectiveness:** AutoReg also outperformed Seasonal Naive by 4.57%, while SARIMA underperformed the seasonal benchmark by 1.37%, demonstrating differences in generalization across forecasting approaches.

#### Evaluating Forecasting Performance Across Horizons

In [14]:
horizon_models = [
    "Naive",
    "Seasonal_Naive",
    "AutoReg",
    "SARIMA",
    "Extra_Trees",
    "Voting_Hybrid"
]

horizon_performance_records = []

for horizon in range(1, forecast_horizon + 1):
    horizon_data = baseline_evaluation_df.loc[
        baseline_evaluation_df["Horizon"] == horizon
    ]

    horizon_actual = horizon_data["Actual"].to_numpy(
        dtype=float
    )

    for model_name in horizon_models:
        horizon_predictions = horizon_data[
            model_name
        ].to_numpy(dtype=float)

        horizon_performance_records.append({
            "Horizon": horizon,
            "Model": model_name,
            "MAE": mean_absolute_error(
                horizon_actual,
                horizon_predictions
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    horizon_actual,
                    horizon_predictions
                )
            ),
            "R2": r2_score(
                horizon_actual,
                horizon_predictions
            ),
            "Forecasts": len(horizon_data)
        })

horizon_performance_df = pd.DataFrame(
    horizon_performance_records
)

assert len(horizon_performance_df) == (
    forecast_horizon * len(horizon_models)
)

assert (
    horizon_performance_df["Forecasts"] == len(test_df)
).all()

horizon_mae_comparison_df = horizon_performance_df.pivot(
    index="Horizon",
    columns="Model",
    values="MAE"
)

horizon_mae_comparison_df = horizon_mae_comparison_df[
    horizon_models
]

horizon_mae_comparison_df["Best_Model"] = (
    horizon_mae_comparison_df[horizon_models]
    .idxmin(axis=1)
)

print("Horizon-Wise MAE Comparison:")
display(
    horizon_mae_comparison_df.style.format(
        {
            model_name: "{:,.2f}"
            for model_name in horizon_models
        }
    )
)

print("\nVoting Hybrid Horizon-Wise Performance:")
display(
    horizon_performance_df.loc[
        horizon_performance_df["Model"] == "Voting_Hybrid"
    ].style.format({
        "MAE": "{:,.2f}",
        "RMSE": "{:,.2f}",
        "R2": "{:.4f}"
    })
)

print("\nHorizon-Wise Forecasting Evaluation: Completed")

Horizon-Wise MAE Comparison:


Model,Naive,Seasonal_Naive,AutoReg,SARIMA,Extra_Trees,Voting_Hybrid,Best_Model
Horizon,,,,,,,
1,"17,617.70","13,889.73","12,527.84","13,735.60","11,671.25","11,850.90",Extra_Trees
2,"19,355.89","13,868.95","12,908.80","13,847.38","11,368.27","11,653.01",Extra_Trees
3,"17,961.16","13,982.54","13,305.07","13,963.18","12,817.23","12,619.77",Voting_Hybrid
4,"19,878.46","13,670.96","13,052.00","13,652.51","12,782.78","12,422.31",Voting_Hybrid
5,"19,087.31","13,761.59","13,325.48","13,850.36","12,784.58","12,665.20",Voting_Hybrid
6,"13,658.13","13,658.13","13,210.23","13,797.00","11,719.57","12,216.19",Extra_Trees
7,"20,886.64","14,751.34","14,797.76","16,072.87","13,479.20","13,830.32",Extra_Trees



Voting Hybrid Horizon-Wise Performance:


,Horizon,Model,MAE,RMSE,R2,Forecasts
5,1,Voting_Hybrid,"11,850.90","16,352.50",0.1430,55
11,2,Voting_Hybrid,"11,653.01","16,131.97",0.1909,55
17,3,Voting_Hybrid,"12,619.77","17,282.58",0.0516,55
23,4,Voting_Hybrid,"12,422.31","17,102.81",0.0649,55
29,5,Voting_Hybrid,"12,665.20","17,235.03",0.0801,55
35,6,Voting_Hybrid,"12,216.19","16,607.72",0.1512,55
41,7,Voting_Hybrid,"13,830.32","18,723.63",-0.0580,55



Horizon-Wise Forecasting Evaluation: Completed


**Key Observations**
- **Horizon-Specific Model Performance:** Extra Trees achieved the lowest MAE at horizons 1, 2, 6, and 7, while Voting Hybrid performed best at horizons 3, 4, and 5, demonstrating complementary forecasting strengths.
- **Forecast Horizon Sensitivity:** Voting Hybrid achieved its lowest MAE of 11,653.01 at horizon 2, while horizon 7 was the most challenging, recording an MAE of 13,830.32 and a negative R² of -0.0580.